In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "datasets" / "raw"

DATASETS = {
    "bottle": RAW_DATA / "bottle",
    "pcb": RAW_DATA / "pcb",
    "road": RAW_DATA / "road",
    "steel": RAW_DATA / "steel",
    "textile": RAW_DATA / "textile",
    "welding": RAW_DATA / "welding",
}

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}

print("Inspectra")
print("=" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw datasets : {RAW_DATA}")
print()

for name, path in DATASETS.items():
    print(
        f"{name:10} -> "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

Inspectra
Project root : d:\Inspectra
Raw datasets : d:\Inspectra\datasets\raw

bottle     -> FOUND
pcb        -> FOUND
road       -> FOUND
steel      -> FOUND
textile    -> FOUND
welding    -> FOUND


In [2]:
def inspect_filesystem(root):
    files = []
    folders = []

    for path in root.rglob("*"):
        if path.is_dir():
            folders.append(path)
        elif path.is_file():
            files.append(path)

    extension_counts = {}

    for file in files:
        extension = file.suffix.lower() or "[no extension]"
        extension_counts[extension] = (
            extension_counts.get(extension, 0) + 1
        )

    return {
        "folders": len(folders),
        "files": len(files),
        "extensions": dict(
            sorted(
                extension_counts.items(),
                key=lambda x: x[1],
                reverse=True
            )
        )
    }


filesystem_results = {}

for name, path in DATASETS.items():

    result = inspect_filesystem(path)

    filesystem_results[name] = result

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(f"Folders : {result['folders']}")
    print(f"Files   : {result['files']}")

    print("\nExtensions:")

    for extension, count in result["extensions"].items():
        print(f"{extension:12} {count}")


BOTTLE
------------------------------------------------------------
Folders : 9
Files   : 17791

Extensions:
.txt         9651
.jpg         8139
.yaml        1

PCB
------------------------------------------------------------
Folders : 9
Files   : 21340

Extensions:
.jpg         10668
.txt         10668
.cache       3
.yaml        1

ROAD
------------------------------------------------------------
Folders : 2
Files   : 40000

Extensions:
.jpg         40000

STEEL
------------------------------------------------------------
Folders : 2
Files   : 18076

Extensions:
.jpg         18074
.csv         2

TEXTILE
------------------------------------------------------------
Folders : 3
Files   : 2473

Extensions:
.jpg         2468
.json        3
.txt         2

WELDING
------------------------------------------------------------
Folders : 35
Files   : 10515

Extensions:
.jpg         5974
.txt         4537
.yaml        3
.md          1


In [3]:
def find_images(root):
    return [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ]


image_inventory = {}

for name, path in DATASETS.items():

    images = find_images(path)

    image_inventory[name] = images

    print(
        f"{name:10} : "
        f"{len(images):,} images"
    )

bottle     : 8,139 images
pcb        : 10,668 images
road       : 40,000 images
steel      : 18,074 images
textile    : 2,468 images
welding    : 5,974 images


In [4]:
def image_dimensions(images, sample_size=500):

    if len(images) > sample_size:
        sampled = random.sample(
            images,
            sample_size
        )
    else:
        sampled = images

    dimensions = []

    for image_path in sampled:

        try:

            with Image.open(image_path) as image:
                width, height = image.size

            dimensions.append(
                {
                    "width": width,
                    "height": height,
                    "path": str(image_path)
                }
            )

        except Exception as error:

            print(
                f"Could not read: {image_path}"
            )
            print(error)

    return pd.DataFrame(dimensions)


dimension_results = {}

for name, images in image_inventory.items():

    df = image_dimensions(images)

    dimension_results[name] = df

    if df.empty:
        continue

    print(f"\n{name.upper()}")
    print("-" * 60)

    print(
        f"Sampled images: {len(df)}"
    )

    print(
        f"Width  : "
        f"{df['width'].min()} - "
        f"{df['width'].max()}"
    )

    print(
        f"Height : "
        f"{df['height'].min()} - "
        f"{df['height'].max()}"
    )

    print(
        f"Most common:\n"
        f"{df[['width', 'height']].value_counts().head(5)}"
    )


BOTTLE
------------------------------------------------------------
Sampled images: 500
Width  : 640 - 640
Height : 640 - 640
Most common:
width  height
640    640       500
Name: count, dtype: int64

PCB
------------------------------------------------------------
Sampled images: 500
Width  : 600 - 601
Height : 600 - 601
Most common:
width  height
600    600       374
601    601       126
Name: count, dtype: int64

ROAD
------------------------------------------------------------
Sampled images: 500
Width  : 227 - 227
Height : 227 - 227
Most common:
width  height
227    227       500
Name: count, dtype: int64

STEEL
------------------------------------------------------------
Sampled images: 500
Width  : 1600 - 1600
Height : 256 - 256
Most common:
width  height
1600   256       500
Name: count, dtype: int64

TEXTILE
------------------------------------------------------------
Sampled images: 500
Width  : 640 - 640
Height : 640 - 640
Most common:
width  height
640    640       500
Nam

In [5]:
ANNOTATION_EXTENSIONS = {
    ".txt",
    ".json",
    ".xml",
    ".csv",
    ".yaml",
    ".yml",
}


def find_annotations(root):

    results = []

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() in ANNOTATION_EXTENSIONS:

            results.append(path)

    return results


annotation_inventory = {}

for name, path in DATASETS.items():

    annotations = find_annotations(path)

    annotation_inventory[name] = annotations

    print(f"\n{name.upper()}")
    print("-" * 60)

    for annotation in annotations[:30]:
        print(annotation.relative_to(path))

    if len(annotations) > 30:
        print(
            f"... and "
            f"{len(annotations) - 30} more"
        )


BOTTLE
------------------------------------------------------------
data.yaml
test\labels\WIN_20240221_11_48_23_Pro_jpg.rf.c9c2e99013358697bdd6095805d6fada.txt
test\labels\WIN_20240221_11_48_33_Pro_jpg.rf.e2a114e025c97e13469e9c06c9c9bd77.txt
test\labels\WIN_20240221_11_48_34_Pro-2-_jpg.rf.4939c0138499f3cfb24ac839482693d1.txt
test\labels\WIN_20240221_11_48_40_Pro_jpg.rf.812e33d2efdc1dc6c461395bf2400792.txt
test\labels\WIN_20240221_11_48_41_Pro-2-_jpg.rf.43d8debb47b4b3bf90489bf56c311ef9.txt
test\labels\WIN_20240221_11_49_04_Pro_jpg.rf.c9284da301c3407e3fdc3e70ac7d255a.txt
test\labels\WIN_20240221_11_49_05_Pro_jpg.rf.a95c95c1391d317f6d1d77548d162465.txt
test\labels\WIN_20240221_11_49_27_Pro-2-_jpg.rf.c459ec223a4641c5c3a1adba89074659.txt
test\labels\WIN_20240221_11_49_27_Pro_jpg.rf.2d4469ea9f4b6b3827fda8d0f8cd96ce.txt
test\labels\WIN_20240221_11_49_28_Pro_jpg.rf.70b4b62f8f8f2d08fe868fe4e08c6160.txt
test\labels\WIN_20240221_11_51_38_Pro_jpg.rf.811c487d9801b48650db6ab735254fe8.txt
test\label

In [6]:
def detect_format(root):

    files = [
        path
        for path in root.rglob("*")
        if path.is_file()
    ]

    extensions = {
        path.suffix.lower()
        for path in files
    }

    has_yaml = (
        ".yaml" in extensions
        or ".yml" in extensions
    )

    has_json = ".json" in extensions
    has_txt = ".txt" in extensions
    has_csv = ".csv" in extensions
    has_xml = ".xml" in extensions

    if has_yaml and has_txt:
        return "YOLO"

    if has_json:
        return "COCO/JSON"

    if has_csv:
        return "CSV-based"

    if has_xml:
        return "XML-based"

    if has_txt:
        return "TXT-based"

    return "Image-only"


format_results = {}

for name, path in DATASETS.items():

    dataset_format = detect_format(path)

    format_results[name] = dataset_format

    print(
        f"{name:10} -> "
        f"{dataset_format}"
    )

bottle     -> YOLO
pcb        -> YOLO
road       -> Image-only
steel      -> CSV-based
textile    -> COCO/JSON
welding    -> YOLO


In [7]:
yaml_files = {}

for name, path in DATASETS.items():

    files = list(path.rglob("*.yaml"))
    files += list(path.rglob("*.yml"))

    yaml_files[name] = files

    print(f"\n{name.upper()}")

    if not files:
        print("No YAML files")

    for file in files:
        print(file.relative_to(path))


BOTTLE
data.yaml

PCB
data.yaml

ROAD
No YAML files

STEEL
No YAML files

TEXTILE
No YAML files

WELDING
1\data.yaml
2\1\data.yaml
2\2\data.yaml


In [8]:
import yaml


yolo_metadata = {}


for name, files in yaml_files.items():

    if not files:
        continue

    print(f"\n{name.upper()}")
    print("-" * 60)

    for file in files:

        try:

            with open(
                file,
                "r",
                encoding="utf-8"
            ) as f:

                data = yaml.safe_load(f)

            yolo_metadata[name] = data

            print(
                json.dumps(
                    data,
                    indent=2,
                    default=str
                )
            )

        except Exception as error:

            print(
                f"Could not parse {file}: {error}"
            )


BOTTLE
------------------------------------------------------------
{
  "train": "../train/images",
  "val": "../valid/images",
  "test": "../test/images",
  "nc": 4,
  "names": [
    "Cap",
    "Missing",
    "Wrong bottle",
    "box"
  ]
}

PCB
------------------------------------------------------------
{
  "path": "../datasets/pcb",
  "train": "train",
  "val": "val",
  "test": "test",
  "names": {
    "0": "mouse_bite",
    "1": "spur",
    "2": "missing_hole",
    "3": "short",
    "4": "open_circuit",
    "5": "spurious_copper"
  }
}

WELDING
------------------------------------------------------------
{
  "train": "../train/images",
  "val": "../valid/images",
  "test": "../test/images",
  "nc": 5,
  "names": [
    "adj",
    "int",
    "geo",
    "pro",
    "non"
  ]
}
{
  "train": "../train/images",
  "val": "../valid/images",
  "test": "../test/images",
  "nc": 3,
  "names": [
    "Bad Weld",
    "Good Weld",
    "Defect"
  ]
}
{
  "train": "../train/images",
  "val": "../v

In [9]:
summary = []

for name in DATASETS:

    summary.append(
        {
            "dataset": name,
            "images": len(
                image_inventory[name]
            ),
            "format": format_results[name],
            "annotation_files": len(
                annotation_inventory[name]
            ),
            "yaml_files": len(
                yaml_files[name]
            )
        }
    )


summary_df = pd.DataFrame(
    summary
)

summary_df

,dataset,images,format,annotation_files,yaml_files
0,bottle,8139,YOLO,9652,1
1,pcb,10668,YOLO,10669,1
2,road,40000,Image-only,0,0
3,steel,18074,CSV-based,2,0
4,textile,2468,COCO/JSON,5,0
5,welding,5974,YOLO,4540,3


In [10]:
def show_samples(
    dataset_name,
    n=12
):

    images = image_inventory[
        dataset_name
    ]

    if not images:
        print("No images found.")
        return

    selected = random.sample(
        images,
        min(n, len(images))
    )

    columns = 4
    rows = int(
        np.ceil(
            len(selected) / columns
        )
    )

    plt.figure(
        figsize=(16, 4 * rows)
    )

    for index, image_path in enumerate(
        selected,
        start=1
    ):

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

            ax = plt.subplot(
                rows,
                columns,
                index
            )

            ax.imshow(image)

            ax.set_title(
                image_path.name
            )

            ax.axis("off")

        except Exception as error:

            print(
                f"Failed: {image_path}"
            )
            print(error)

    plt.tight_layout()
    plt.show()